# Modulo 07 - Moduli e Pacchetti

---

Finora, tutto il codice che hai scritto ha vissuto all'interno di un unico notebook. In questo modulo imparerai a **riutilizzare il codice tra notebook diversi**, partendo dai modi per importare codice già pronto nel tuo programma con `import`, `from` e `as`. Successivamente, creerai il tuo primo **modulo** — un file `.py` con una classe che legge file CSV — e poi organizzerai moduli correlati all'interno di un **pacchetto**, una cartella che raggruppa file che lavorano insieme. Infine, conoscerai il **PyPI** e il **pip**, gli strumenti che danno accesso a migliaia di pacchetti creati dalla comunità Python, e userai il pacchetto `requests` per interrogare un servizio su internet. Al termine di questo modulo, capirai come lo stesso Python — e librerie come Pandas e NumPy, che vedrai più avanti nel corso — siano organizzati internamente.

Corso: Ready To Deploy

Creato da: [Enzo Schitini](https://www.linkedin.com/in/enzoschitini)

---

## Argomenti

| **Argomento** | Descrizione |
| --- | --- |
| 1. from / import / as | Come importare un modulo intero, solo alcuni suoi nomi, o assegnare loro un alias. |
| 2. Moduli | Cos'è un modulo, come crearne uno proprio e importare codice riutilizzabile tra notebook. |
| 3. Pacchetti | Come organizzare più moduli correlati all'interno di una cartella (pacchetto) e importare codice al suo interno. |
| 4. Scaricare pacchetti | Come trovare, installare e usare pacchetti di terze parti con PyPI e pip, applicato a una richiesta HTTP con il pacchetto requests. |

---

## 1. from / import / as

Python viene fornito con una **libreria standard** enorme — la cosiddetta filosofia "*batterie incluse*" — piena di moduli pronti per compiti comuni (vedi l'elenco completo [qui](https://docs.python.org/3/library/)). Per usare uno qualsiasi di essi, occorre prima **importarlo**.

### 1.1 import

Il modo più semplice è `import nome_del_modulo`, che porta l'intero modulo. Dopo, ogni funzione o variabile viene raggiunta con `nome_del_modulo.nome_elemento`.

**Esempio:** il modulo `random`, per generare valori e scelte casuali.

In [1]:
import random

In [2]:
opzioni = ['sasso', 'carta', 'forbici']

scelta_del_computer = random.choice(opzioni)
print(scelta_del_computer)

tesoura


In [3]:
numero_casuale = random.random()  # float casuale nell'intervallo [0, 1)
print(numero_casuale)

0.5191044337047542


**Esempio:** il modulo `math`, con funzioni e costanti matematiche.

In [4]:
import math

In [5]:
potenza = math.pow(10, 3)
print(potenza)

1000.0


In [6]:
# math.ceil() arrotonda sempre PER ECCESSO, a differenza di round() visto nel Modulo 01
numero_arrotondato = math.ceil(10.1)
print(numero_arrotondato)

11


In [7]:
print(math.pi)

3.141592653589793


### 1.2 from / import

Quando ci servono solo uno o alcuni elementi specifici di un modulo, importiamo esattamente quei nomi con `from modulo import nome`. Il vantaggio è usare il nome direttamente, senza il prefisso del modulo.

```python
from modulo import nome
```

> ⚠️ **Attenzione:** resta disponibile solo ciò che è stato esplicitamente importato. Se il modulo `time` ha le funzioni `time()` e `sleep()`, ma importiamo solo `time`, provare a usare `sleep()` genera un `NameError` (visto nel Modulo 03) — semplicemente non esiste nel notebook.

In [8]:
from time import time

# 'time' è stato importato e funziona
print(time())  # secondi trascorsi dal 01/01/1970 (l'"epoca Unix")

# sleep(1)  # ❌ NameError: name 'sleep' is not defined (sleep non è stato importato)

1789135666.9954908


Per importare più di un nome dallo stesso modulo, li separiamo con la virgola:

In [9]:
from time import time, sleep

istante_iniziale = time()
sleep(1)  # mette in pausa l'esecuzione del programma per 1 secondo
istante_finale = time()

print(f'Tempo trascorso: {istante_finale - istante_iniziale:.2f} secondi')

Tempo decorrido: 1.00 segundos


### 1.3 from / import / as

Possiamo assegnare un alias (rinominare) a un modulo o a un elemento importato con `as`. È utile per accorciare nomi lunghi o evitare conflitti con altri nomi già usati nel codice.

**Esempio:** la classe `datetime`, che vive dentro il modulo omonimo, `datetime` — a cui diamo l'alias `dt` per non confondere i due.

In [10]:
from datetime import datetime as dt

In [11]:
print(dt.now())

2026-09-11 11:07:48.102980


In [12]:
print(dt.now().day)

11


In [13]:
print(dt.now().year)

2026


> 💡 **Suggerimento:** alias brevi e consacrati dal mercato — come `import pandas as pd` e `import numpy as np` — sono una convenzione che troverai in praticamente ogni codice di analisi dati in Python, anche nei prossimi moduli di questo corso.

---

## 2. Moduli

Un **modulo** è semplicemente un file `.py` contenente codice Python — funzioni, classi, variabili — pronto per essere riutilizzato. I moduli importati nella sezione 1 fanno parte della libreria standard di Python; in questa sezione ne creerai uno tuo.

### 2.1 Motivazione

Hai scritto una classe capace di leggere un file CSV ed estrarre una qualsiasi delle sue colonne. È utile e ti servirà in diversi notebook — non ha senso copiare e incollare lo stesso codice ogni volta.

> 💡 **Suggerimento:** gli `: str` e `: int` dopo i parametri qui sotto sono *type hint* — annotazioni opzionali che documentano il tipo atteso di ciascun argomento. Python non le impone, ma rendono il codice più facile da capire.

In [14]:
# Legge un file CSV e permette di estrarre colonne specifiche
class ArquivoCSV:

  def __init__(self, caminho_arquivo: str):
    self.caminho_arquivo = caminho_arquivo
    self.linhas = self._ler_linhas()
    self.colunas = self._extrair_nomes_colunas()

  def _ler_linhas(self):
    with open(self.caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
      return arquivo.readlines()

  def _extrair_nomes_colunas(self):
    return self.linhas[0].strip().split(',')

  def extrair_coluna(self, indice_coluna: int):
    valores = []
    for linha in self.linhas[1:]:  # [1:] salta la riga di intestazione
      valores.append(linha.strip().split(',')[indice_coluna])
    return valores

Un file `banco.csv` per testare la classe:

In [15]:
conteudo_banco_csv = '''age,job,marital,education,default,balance,housing,loan
30,unemployed,married,primary,no,1787,no,no
33,services,married,secondary,no,4789,yes,yes
35,management,single,tertiary,no,1350,yes,no
30,management,married,tertiary,no,1476,yes,yes
59,blue-collar,married,secondary,no,0,yes,no
35,management,single,tertiary,no,747,no,no
36,self-employed,married,tertiary,no,307,yes,no
39,technician,married,secondary,no,147,yes,no
41,entrepreneur,married,tertiary,no,221,yes,no
43,services,married,primary,no,-88,yes,yes
'''

with open('banco.csv', mode='w', encoding='utf-8') as arquivo:
  arquivo.write(conteudo_banco_csv)

> 💡 **Suggerimento:** in Google Colab, il comando magico `%%writefile nome_del_file` fa esattamente la stessa cosa in un'unica cella. Qui usiamo `open()` perché funziona in qualsiasi ambiente Python, non solo su Colab.

In [16]:
arquivo_banco = ArquivoCSV(caminho_arquivo='banco.csv')

escolaridade = arquivo_banco.extrair_coluna(indice_coluna=3)
print(escolaridade)

['primary', 'secondary', 'tertiary', 'tertiary', 'secondary', 'tertiary', 'tertiary', 'secondary', 'tertiary', 'primary']


### 2.2 Definizione

Un modulo proprio si crea esattamente come il file di dati qui sopra: scriviamo il codice in un file `.py`. Python trova automaticamente quel file quando si trova nella stessa cartella del notebook — basta importarlo per nome, senza l'estensione.

> ⚠️ **Attenzione:** la cella qui sotto scrive il codice della classe in un file `arquivo_csv.py` **a partire da una stringa**, solo perché questo notebook possa creare il modulo da solo, dall'inizio alla fine. Nel tuo lavoro quotidiano, scriverai `arquivo_csv.py` direttamente in un editor di codice — e non lo genererai a partire da un'altra cella.

In [17]:
codigo_do_modulo_csv = '''# Legge un file CSV e permette di estrarre colonne specifiche
class ArquivoCSV:

    def __init__(self, caminho_arquivo: str):
        self.caminho_arquivo = caminho_arquivo
        self.linhas = self._ler_linhas()
        self.colunas = self._extrair_nomes_colunas()

    def _ler_linhas(self):
        with open(self.caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
            return arquivo.readlines()

    def _extrair_nomes_colunas(self):
        return self.linhas[0].strip().split(',')

    def extrair_coluna(self, indice_coluna: int):
        valores = []
        for linha in self.linhas[1:]:
            valores.append(linha.strip().split(',')[indice_coluna])
        return valores
'''

with open('arquivo_csv.py', mode='w', encoding='utf-8') as arquivo:
  arquivo.write(codigo_do_modulo_csv)

print('Modulo arquivo_csv.py creato.')

Módulo arquivo_csv.py criado.


### 2.3 Rivisitando la motivazione

Ora importiamo la classe **dal modulo** che abbiamo appena creato, invece di averla definita direttamente nel notebook:

In [18]:
from arquivo_csv import ArquivoCSV

arquivo_banco_modulo = ArquivoCSV(caminho_arquivo='banco.csv')

escolaridade = arquivo_banco_modulo.extrair_coluna(indice_coluna=3)
print(escolaridade)

['primary', 'secondary', 'tertiary', 'tertiary', 'secondary', 'tertiary', 'tertiary', 'secondary', 'tertiary', 'primary']


Il risultato è identico a quello della sezione 2.1 — la differenza è che questa classe ora può essere riutilizzata in **qualsiasi** notebook che si trovi nella stessa cartella, basta importarla.

Sono accessibili solo i nomi che esistono davvero dentro il modulo. Provare a usare un metodo mai definito genera un `AttributeError` (visto nel Modulo 03):

In [19]:
try:
  soma = arquivo_banco_modulo._somar_saldos(coluna='balance')
except AttributeError as exc:
  print(f'Errore: {exc}')

Erro: 'ArquivoCSV' object has no attribute '_somar_saldos'


---

## 3. Pacchetti

Man mano che un progetto cresce, è comune ritrovarsi con diversi moduli correlati. Un **pacchetto** è una cartella che raggruppa questi moduli, permettendo di organizzarli e importarli da un unico percorso.

### 3.1 Motivazione

Ora devi anche elaborare file di testo semplice, come il contenuto di una notizia. Creiamo un'altra classe per questo, simile ad `ArquivoCSV`.

In [20]:
# Legge un file di testo e permette di estrarre una riga specifica
class ArquivoTXT:

  def __init__(self, caminho_arquivo: str):
    self.caminho_arquivo = caminho_arquivo
    self.linhas = self._ler_linhas()

  def _ler_linhas(self):
    with open(self.caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
      return arquivo.readlines()

  def extrair_linha(self, numero_linha: int):
    return self.linhas[numero_linha - 1].strip()

In [21]:
conteudo_noticia = '''Ready To Deploy lancia un modulo sull'organizzazione del codice in Python
Il corso insegna a strutturare progetti più grandi usando moduli e pacchetti, seguendo le migliori pratiche più diffuse nel mercato.
'''

with open('noticia.txt', mode='w', encoding='utf-8') as arquivo:
  arquivo.write(conteudo_noticia)

In [22]:
arquivo_noticia = ArquivoTXT(caminho_arquivo='noticia.txt')

titulo = arquivo_noticia.extrair_linha(numero_linha=1)
print(titulo)

Ready To Deploy lança módulo sobre organização de código em Python


### 3.2 Definizione

Un pacchetto è una **cartella comune**, con un dettaglio: tradizionalmente contiene un file `__init__.py` (può essere vuoto) che segnala a Python che quella cartella va trattata come un pacchetto importabile.

> 💡 **Suggerimento:** a partire da Python 3.3, l'interprete riconosce anche cartelle senza `__init__.py` come pacchetti "impliciti" (*namespace packages*). Ciononostante, includere il file resta la pratica più comune e più compatibile tra le versioni.

Creiamo un pacchetto chiamato `arquivo` e spostiamo i moduli `arquivo_csv.py` e `arquivo_txt.py` al suo interno.

In [23]:
import os

os.makedirs('arquivo', exist_ok=True)

# File vuoto che segna la cartella come un pacchetto
with open('arquivo/__init__.py', mode='w', encoding='utf-8') as arquivo:
  arquivo.write('')

print('Pacchetto "arquivo" creato.')

Pacote "arquivo" criado.


Riutilizziamo lo stesso codice del modulo `ArquivoCSV`, scritto nella sezione 2.2, ora salvato dentro il pacchetto:

In [24]:
with open('arquivo/arquivo_csv.py', mode='w', encoding='utf-8') as arquivo:
  arquivo.write(codigo_do_modulo_csv)

print('arquivo/arquivo_csv.py creato.')

arquivo/arquivo_csv.py criado.


In [25]:
codigo_do_modulo_txt = '''# Legge un file di testo e permette di estrarre una riga specifica
class ArquivoTXT:

    def __init__(self, caminho_arquivo: str):
        self.caminho_arquivo = caminho_arquivo
        self.linhas = self._ler_linhas()

    def _ler_linhas(self):
        with open(self.caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
            return arquivo.readlines()

    def extrair_linha(self, numero_linha: int):
        return self.linhas[numero_linha - 1].strip()
'''

with open('arquivo/arquivo_txt.py', mode='w', encoding='utf-8') as arquivo:
  arquivo.write(codigo_do_modulo_txt)

print('arquivo/arquivo_txt.py creato.')

arquivo/arquivo_txt.py criado.


### 3.3 Rivisitando la motivazione

Ora importiamo i due moduli **da dentro il pacchetto**, usando `.` per indicare il percorso cartella | modulo:

In [26]:
from arquivo.arquivo_csv import ArquivoCSV
from arquivo.arquivo_txt import ArquivoTXT

In [27]:
arquivo_banco_pacote = ArquivoCSV(caminho_arquivo='banco.csv')

escolaridade = arquivo_banco_pacote.extrair_coluna(indice_coluna=3)
print(escolaridade)

['primary', 'secondary', 'tertiary', 'tertiary', 'secondary', 'tertiary', 'tertiary', 'secondary', 'tertiary', 'primary']


In [28]:
arquivo_noticia_pacote = ArquivoTXT(caminho_arquivo='noticia.txt')

titulo = arquivo_noticia_pacote.extrair_linha(numero_linha=1)
print(titulo)

Ready To Deploy lança módulo sobre organização de código em Python


> 💡 Il risultato è lo stesso di prima, ma ora i due moduli sono organizzati dentro un'unica cartella — esattamente come le librerie vere (Pandas, NumPy...), che sono pacchetti con decine di moduli interni.

---

## 4. Scaricare pacchetti

Oltre alla libreria standard e ai moduli che scrivi tu stesso, esiste un universo di pacchetti creati dalla comunità Python per praticamente qualsiasi compito.

### 4.1 PyPI

Il **PyPI** (*Python Package Index*, [pypi.org](https://pypi.org/)) è il repository ufficiale dei pacchetti Python. Qui trovi librerie per analisi dati, automazione, sviluppo web, intelligenza artificiale e molto altro — ognuna con la propria pagina, le proprie versioni e la propria documentazione.

### 4.2 pip

Il **pip** è lo strumento ufficiale per installare pacchetti dal PyPI direttamente dal terminale (su Colab, in una cella con `!` davanti, che esegue un comando di sistema invece che Python).

| Comando | Cosa fa |
| --- | --- |
| `pip install <pacchetto>` | Installa la versione più recente del pacchetto |
| `pip install <pacchetto>==<versione>` | Installa una versione specifica |
| `pip freeze` | Elenca tutti i pacchetti installati e le loro versioni |
| `pip uninstall <pacchetto>` | Rimuove un pacchetto installato |

In [29]:
!pip install requests==2.32.3

  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
Using cached requests-2.32.3-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-airflow-core 3.1.3 requires pyjwt>=2.10.0, but you have pyjwt 2.9.0 which is incompatible.
langchain-huggingface 0.3.1 requires langchain-core<1.0.0,>=0.3.70, but you have langchain-core 1.2.5 which is incompatible.
pinecone-plugin-assistant 1.8.0 requires packaging<25.0,>=24.2, but you have packaging 25.0 which is incompatible.
streamlit 1.38.0 requires packaging<25,>=20, but you have packaging 25.0 which is incompatible.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.3 which is incompatible.
tensorflow-intel 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.20.0 which is incompatible.
transformers 4.57.1 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.20.3 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.

In [30]:
!pip freeze

a2wsgi==1.10.10
absl-py==2.1.0
accelerate==1.10.1
acres==0.5.0
# Editable install with no version control (agent==0.0.1)
-e c:\users\schit\better-ai\my-app
agno==2.1.1
aiofiles==24.1.0
aiohappyeyeballs==2.4.6
aiohttp==3.11.12
aiohttp-retry==2.9.1
aiosignal==1.3.2
aiosmtplib==5.0.0
aiosqlite==0.21.0
alembic==1.17.2
altair==5.4.1
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.9.0
apache-airflow-core==3.1.3
apache-airflow-providers-common-compat==1.10.1
apache-airflow-providers-common-io==1.7.0
apache-airflow-providers-common-sql==1.30.1
apache-airflow-providers-smtp==2.4.1
apache-airflow-providers-standard==1.10.1
apache-airflow-task-sdk==1.1.3
appdirs==1.4.4
argcomplete==3.6.3
arxiv==2.2.0
asgiref==3.10.0
asttokens==2.4.1
astunparse==1.6.3
attrs==24.2.0
azure-core==1.35.1
azure-storage-blob==12.26.0
azure-storage-file-datalake==12.21.0
babel==2.17.0
backoff==2.2.1
bcrypt==4.3.0
beautifulsoup4==4.13.3
blinker==1.8.2
blockbuster==1.5.26
boto3==1.40.45
botocore==1.40.45
bs4=

> 💡 **Suggerimento:** fissare la versione (`==2.32.3`) evita che un aggiornamento futuro del pacchetto rompa un codice che stava già funzionando — una buona pratica nei progetti reali.

### 4.3 requests

Il pacchetto [requests](https://pypi.org/project/requests/) semplifica l'esecuzione di richieste al protocollo web **HTTP**, usato praticamente da ogni servizio su internet.

**Esempio:** stai automatizzando la compilazione di una scheda anagrafica e, a partire dal CAP fornito dal cliente, devi scoprire via, quartiere e città. Il [ViaCEP](https://viacep.com.br/) è un servizio pubblico e gratuito brasiliano per questo tipo di ricerca.

In [31]:
import requests

resposta = requests.get('https://viacep.com.br/ws/01310930/json/')
print(f'Status code: {resposta.status_code}')

Status code: 200


Uno `status_code` uguale a `200` indica che la richiesta è andata a buon fine. Il contenuto restituito arriva come testo, nel formato **JSON** — un formato molto usato per lo scambio di dati tra sistemi:

In [32]:
print(resposta.text)

{
  "cep": "01310-930",
  "logradouro": "Avenida Paulista",
  "complemento": "2100",
  "unidade": "Banco Safra S.A",
  "bairro": "Bela Vista",
  "localidade": "São Paulo",
  "uf": "SP",
  "estado": "São Paulo",
  "regiao": "Sudeste",
  "ibge": "3550308",
  "gia": "1004",
  "ddd": "11",
  "siafi": "7107"
}


Il modulo `json`, anch'esso della libreria standard, converte questo testo in un dizionario Python con `json.loads()`:

In [33]:
import json

endereco = json.loads(resposta.text)
print(endereco)

{'cep': '01310-930', 'logradouro': 'Avenida Paulista', 'complemento': '2100', 'unidade': 'Banco Safra S.A', 'bairro': 'Bela Vista', 'localidade': 'São Paulo', 'uf': 'SP', 'estado': 'São Paulo', 'regiao': 'Sudeste', 'ibge': '3550308', 'gia': '1004', 'ddd': '11', 'siafi': '7107'}


> 💡 Lo stesso `requests` offre già una scorciatoia per questo, senza dover importare `json` manualmente: `resposta.json()`.

In [34]:
endereco = resposta.json()

print(f"Rua: {endereco['logradouro']}")
print(f"Bairro: {endereco['bairro']}")
print(f"Cidade: {endereco['localidade']} - {endereco['uf']}")

Rua: Avenida Paulista
Bairro: Bela Vista
Cidade: São Paulo - SP


> 💡 Praticamente ogni pacchetto che si connette a un'API (social network, banche, sistemi di pagamento, lo stesso ChatGPT...) segue questa stessa logica: una richiesta HTTP che restituisce dati in JSON.

---

## Riepilogo del Modulo

| Concetto | Cos'è | Sintassi |
| --- | --- | --- |
| Importare un modulo intero | Porta tutto il contenuto, raggiunto con prefisso | `import modulo` |
| Importare nomi specifici | Porta solo ciò che è stato richiesto, senza prefisso | `from modulo import nome` |
| Assegnare un alias all'importazione | Rinomina il modulo o il nome importato | `import modulo as alias` |
| Modulo | Un file `.py` con codice riutilizzabile | `arquivo_csv.py` |
| Pacchetto | Una cartella con moduli correlati e un `__init__.py` | `arquivo/arquivo_csv.py` |
| PyPI | Repository ufficiale dei pacchetti di terze parti | [pypi.org](https://pypi.org/) |
| pip | Strumento per installare, elencare e rimuovere pacchetti | `pip install`, `pip freeze`, `pip uninstall` |

Pacchetti visti in questo modulo: `random`, `math`, `time`, `datetime`, `os`, `json` (libreria standard) e `requests` (di terze parti, via PyPI).